# Codex Python SDK — Special Use-Case Validation Notebook

Goal: run many SDK-specific use cases and collect **where/why failures happen**.

This notebook is a validation harness, not a tutorial.


In [ ]:
# Bootstrap current kernel with local SDK + deps.
import sys, subprocess
from pathlib import Path

repo_python_dir = Path.cwd().resolve().parent
src_dir = repo_python_dir / 'src'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo_python_dir)])
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print('Kernel:', sys.executable)
print('SDK src:', src_dir)


In [ ]:
from __future__ import annotations

import asyncio
import traceback
from dataclasses import dataclass, asdict
from typing import Any

from codex_app_server import Codex, TextInput, ImageInput, LocalImageInput
from codex_app_server.retry import retry_on_overload
from codex_app_server.async_client import AsyncAppServerClient


In [ ]:
@dataclass
class CaseResult:
    name: str
    status: str
    detail: str = ''
    error_type: str = ''
    error_message: str = ''
    traceback_tail: str = ''


def run_case(name: str, fn):
    try:
        detail = fn()
        if detail is None:
            detail = ''
        return CaseResult(name=name, status='PASS', detail=str(detail)[:500])
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()
        tail = '
'.join(tb[-8:])
        return CaseResult(
            name=name,
            status='FAIL',
            error_type=type(e).__name__,
            error_message=str(e),
            traceback_tail=tail,
        )


## Define special SDK use cases


In [ ]:
from base64 import b64decode
from pathlib import Path

sample_img = Path.cwd() / 'sample.png'
if not sample_img.exists():
    sample_img.write_bytes(
        b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO7Z4xQAAAAASUVORK5CYII=')
    )


def case_sync_basic_turn():
    with Codex() as codex:
        thread = codex.thread_start(model='gpt-5')
        result = thread.turn(TextInput('Return exactly: SDK_OK')).run()
        return f"status={result.status}, text={result.text!r}"


def case_sync_retry_wrapper():
    with Codex() as codex:
        thread = codex.thread_start(model='gpt-5')
        result = retry_on_overload(
            lambda: thread.turn(TextInput('3 bullets about retries')).run(),
            max_attempts=3,
            initial_delay_s=0.25,
            max_delay_s=2.0,
        )
        return f"status={result.status}"


def case_thread_resume_flow():
    with Codex() as codex:
        t = codex.thread_start(model='gpt-5')
        first = t.turn(TextInput('One fact about Saturn')).run()
        resumed = codex.thread(first.thread_id)
        second = resumed.turn(TextInput('One more fact')).run()
        return f"second_status={second.status}"


def case_thread_read_list():
    with Codex() as codex:
        t = codex.thread_start(model='gpt-5')
        _ = codex.thread_list(limit=10)
        _ = codex.thread_read(t.id, include_turns=False)
        return 'list+read ok'


def case_turn_steer_best_effort():
    with Codex() as codex:
        t = codex.thread_start(model='gpt-5')
        r = t.turn(TextInput('Start brief answer')).run()
        try:
            _ = codex.turn_steer(t.id, r.turn_id, TextInput('Add one sentence'))
            return 'steer ok'
        except Exception as e:
            return f'steer skipped: {type(e).__name__}: {e}'


def case_image_remote_best_effort():
    with Codex() as codex:
        t = codex.thread_start(model='gpt-5')
        r = t.turn([
            TextInput('Describe this image in 2 bullets'),
            ImageInput('https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg'),
        ]).run()
        return f"status={r.status}"


def case_image_local():
    with Codex() as codex:
        t = codex.thread_start(model='gpt-5')
        r = t.turn([
            TextInput('Describe local image in 2 bullets'),
            LocalImageInput(str(sample_img)),
        ]).run()
        return f"status={r.status}"


def case_error_mapping_missing_method():
    with Codex() as codex:
        try:
            codex._client.request('demo/missingMethod', {})  # noqa: SLF001
        except Exception as e:
            return f"caught={type(e).__name__}: {e}"
        return 'unexpected: no error'


async def _async_one(prompt: str):
    async with AsyncAppServerClient() as client:
        await client.initialize()
        started = await client.thread_start(model='gpt-5')
        tid = started['thread']['id']
        turn = await client.turn_text(tid, prompt)
        turn_id = turn['turn']['id']
        chunks=[]
        while True:
            evt = await client.next_notification()
            if evt.method == 'item/agentMessage/delta':
                chunks.append((evt.params or {}).get('delta',''))
            if evt.method == 'turn/completed' and (evt.params or {}).get('turn',{}).get('id') == turn_id:
                break
        return ''.join(chunks).strip()


def case_async_concurrent_fanout():
    async def run_all():
        prompts = [
            'one line on retries',
            'one line on backpressure',
            'one line on idempotency',
        ]
        return await asyncio.gather(*(_async_one(p) for p in prompts), return_exceptions=True)

    outs = asyncio.run(run_all())
    return f"outputs={outs}"


## Execute all cases and build failure report


In [ ]:
cases = [
    ('sync_basic_turn', case_sync_basic_turn),
    ('sync_retry_wrapper', case_sync_retry_wrapper),
    ('thread_resume_flow', case_thread_resume_flow),
    ('thread_read_list', case_thread_read_list),
    ('turn_steer_best_effort', case_turn_steer_best_effort),
    ('image_remote_best_effort', case_image_remote_best_effort),
    ('image_local', case_image_local),
    ('error_mapping_missing_method', case_error_mapping_missing_method),
    ('async_concurrent_fanout', case_async_concurrent_fanout),
]

results = [run_case(name, fn) for name, fn in cases]
rows = [asdict(r) for r in results]

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df[['name','status','detail','error_type','error_message']])
except Exception:
    for r in rows:
        print(r)

passed = sum(1 for r in results if r.status == 'PASS')
failed = [r for r in results if r.status == 'FAIL']
print(f'PASS={passed} FAIL={len(failed)} TOTAL={len(results)}')


In [ ]:
# Failure-focused debug report (where and why)
for r in [x for x in results if x.status == 'FAIL']:
    print('
' + '='*80)
    print('CASE:', r.name)
    print('ERROR TYPE:', r.error_type)
    print('ERROR MESSAGE:', r.error_message)
    print('TRACEBACK TAIL:
', r.traceback_tail)
